In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:

from pathlib import Path
 
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
!pip install -q sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 76.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incomp

In [4]:
import pandas as pd
import numpy as np

from collections import Counter, defaultdict

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

from sentence_transformers import SentenceTransformer

In [5]:
# ============================================================
# LOAD DATA
# ============================================================

TRAIN_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
TEST_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

print(train.shape)
print(test.shape)

(2000, 8)
(500, 7)


In [6]:
LABELS = ["A","B","C","D","E"]

# ============================================================
# OPTION KEY
# ============================================================

def create_option_key(df):

    return (
        df["A"].astype(str) + "||" +
        df["B"].astype(str) + "||" +
        df["C"].astype(str) + "||" +
        df["D"].astype(str) + "||" +
        df["E"].astype(str)
    )

train["option_key"] = create_option_key(train)
test["option_key"] = create_option_key(test)

In [7]:
# ============================================================
# EXACT MATCH MEMORY
# ============================================================

memory = {}

for key, grp in train.groupby("option_key"):

    counts = Counter(grp["answer"])

    ranking = [
        x[0]
        for x in sorted(
            counts.items(),
            key=lambda x: x[1],
            reverse=True
        )
    ]

    memory[key] = ranking

print("Unique option sets:", len(memory))

Unique option sets: 588


In [8]:
# ============================================================
# TEXT CREATION
# ============================================================

def build_text(df):

    return (
        df["prompt"].fillna("").astype(str)
        + " [SEP] "
        + df["A"].fillna("").astype(str)
        + " [SEP] "
        + df["B"].fillna("").astype(str)
        + " [SEP] "
        + df["C"].fillna("").astype(str)
        + " [SEP] "
        + df["D"].fillna("").astype(str)
        + " [SEP] "
        + df["E"].fillna("").astype(str)
    )

train_text = build_text(train)
test_text = build_text(test)

In [9]:
# ============================================================
# TF-IDF RETRIEVAL
# ============================================================

print("Building TFIDF...")

vectorizer = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1,2),
    stop_words="english"
)

X_train = vectorizer.fit_transform(train_text)
X_test = vectorizer.transform(test_text)

nn = NearestNeighbors(
    n_neighbors=20,
    metric="cosine"
)

nn.fit(X_train)

tfidf_distances, tfidf_indices = nn.kneighbors(X_test)

# ============================================================
# TFIDF PREDICTION
# ============================================================

def tfidf_predict(idx):

    scores = defaultdict(float)

    neighbors = tfidf_indices[idx]
    dists = tfidf_distances[idx]

    for n_idx, dist in zip(neighbors, dists):

        similarity = 1 - dist

        label = train.iloc[n_idx]["answer"]

        scores[label] += similarity

    ranked = sorted(
        scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    labels = [x[0] for x in ranked]

    for lab in LABELS:
        if lab not in labels:
            labels.append(lab)

    return labels[:3]

Building TFIDF...


In [10]:
# ============================================================
# BGE EMBEDDINGS
# ============================================================

print("Loading BGE model...")

bge_model = SentenceTransformer(
    "BAAI/bge-small-en-v1.5"
)

print("Encoding train...")

train_embeddings = bge_model.encode(
    train_text.tolist(),
    batch_size=64,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Encoding test...")

test_embeddings = bge_model.encode(
    test_text.tolist(),
    batch_size=64,
    normalize_embeddings=True,
    show_progress_bar=True
)

Loading BGE model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Encoding train...


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Encoding test...


Batches:   0%|          | 0/8 [00:00<?, ?it/s]

In [11]:
# ============================================================
# BGE RETRIEVAL
# ============================================================

TOPK = 20

def bge_predict(test_emb):

    sims = np.dot(train_embeddings, test_emb)

    idx = np.argsort(-sims)[:TOPK]

    scores = defaultdict(float)

    for i in idx:

        label = train.iloc[i]["answer"]

        scores[label] += sims[i]

    ranked = sorted(
        scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    labels = [x[0] for x in ranked]

    for lab in LABELS:
        if lab not in labels:
            labels.append(lab)

    return labels[:3]

In [12]:
# ============================================================
# ENSEMBLE
# ============================================================

predictions = []

exact_match_hits = 0

for i, row in test.iterrows():

    key = row["option_key"]

    # ========================================================
    # EXACT MATCH
    # ========================================================

    if key in memory:

        exact_match_hits += 1

        ranking = memory[key].copy()

        for lab in LABELS:
            if lab not in ranking:
                ranking.append(lab)

        predictions.append(
            " ".join(ranking[:3])
        )

        continue

    # ========================================================
    # TFIDF
    # ========================================================

    tfidf_top3 = tfidf_predict(i)

    # ========================================================
    # BGE
    # ========================================================

    bge_top3 = bge_predict(
        test_embeddings[i]
    )

    # ========================================================
    # WEIGHTED VOTE
    # ========================================================

    scores = defaultdict(float)

    tfidf_weights = [3,2,1]
    bge_weights = [3,2,1]

    for rank, label in enumerate(tfidf_top3):
        scores[label] += tfidf_weights[rank]

    for rank, label in enumerate(bge_top3):
        scores[label] += bge_weights[rank]

    ranked = sorted(
        scores,
        key=scores.get,
        reverse=True
    )

    predictions.append(
        " ".join(ranked[:3])
    )

print("Exact Match Hits:", exact_match_hits)

Exact Match Hits: 458


In [13]:
# ============================================================
# SUBMISSION
# ============================================================

submission = pd.DataFrame({
    "ID": test["id"],
    "Prediction": predictions
})

submission.to_csv(
    "submission.csv",
    index=False
)

print(submission.head())
print("\nSaved submission.csv")

   ID Prediction
0   1      A B C
1   2      B A C
2   3      B A C
3   4      E A B
4   5      C A B

Saved submission.csv
